# CPU cache and COLMAP track audit

Use a **CPU High-RAM** runtime and choose **Runtime -> Run all**. This free preflight creates/restores the exact frame selection, checks the owned COLMAP cache, filters invalid tracks, and publishes the receipt required by the A100 notebook.

In [ ]:
INPUT_FOLDER = ""  # @param {type:"string"}


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive').resolve()


In [ ]:
import json, unicodedata
from pathlib import Path, PurePosixPath
raw_folder = INPUT_FOLDER.strip()
if not raw_folder:
    raw_folder = input('MyDrive-relative input folder: ').strip()
assert raw_folder and '\\' not in raw_folder, 'Use a MyDrive-relative POSIX path'
assert not any(unicodedata.category(ch) == 'Cc' for ch in raw_folder)
folder = PurePosixPath(raw_folder)
assert not folder.is_absolute() and folder.parts
assert all(part not in {'', '.', '..'} for part in folder.parts)
assert not raw_folder.endswith(('_result', '_learned_test_result', '_learned_test_diagnostics', '_learned_test_cache'))
INPUT_PATH = DRIVE_ROOT.joinpath(*folder.parts).resolve()
INPUT_PATH.relative_to(DRIVE_ROOT)
assert INPUT_PATH.is_dir(), f'Input folder does not exist: {INPUT_PATH}'
CACHE_PATH = INPUT_PATH.with_name(INPUT_PATH.name + '_learned_test_cache')
RUN_SPEC = {'schema_version': 1, 'input_folder': folder.as_posix(), 'publish': {'replace_owned_result': True}}
SPEC_PATH = Path('/content/learned_spec.json')
with SPEC_PATH.open('w', encoding='utf-8') as handle:
    json.dump(RUN_SPEC, handle, sort_keys=True, separators=(',', ':'))
print(f'Input: {INPUT_PATH}')
print(f'CPU audit/cache folder: {CACHE_PATH}')


In [ ]:
import shutil, subprocess
from pathlib import Path
SOURCE_ROOT = Path('/content/gaussian-splatter-src')
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
REPOSITORY_URL = 'https://github.com/mehmettahacumurcu/gaussian-splatter.git'
COMMIT_SHA = '7366421c3db380971205dbad5bbb679a47a51f53'
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(SOURCE_ROOT)], check=True)
subprocess.run(['git', '-C', str(SOURCE_ROOT), 'checkout', '--detach', COMMIT_SHA], check=True)
actual = subprocess.run(['git', '-C', str(SOURCE_ROOT), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
assert actual == COMMIT_SHA, 'Immutable source checkout mismatch'


In [ ]:
import subprocess, sys
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26,<2.0', 'opencv-python-headless>=4.8',
    'Pillow>=10', 'pydantic>=2.6,<3',
], check=True)
print('CPU audit dependencies are ready.')


In [ ]:
import json, subprocess, sys, traceback
from pathlib import Path
from google.colab import drive, runtime
failure = None
try:
    completed = subprocess.run(
        [
            sys.executable, "-u", "-m",
            "scripts.learned_quality_cache_audit",
            "--spec", "/content/learned_spec.json",
        ],
        cwd=SOURCE_ROOT, check=False,
    )
    receipt_path = Path('/content/learned_audit_result.json')
    if not receipt_path.is_file():
        raise RuntimeError('CPU audit ended without a receipt')
    receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
    if completed.returncode != 0 or receipt.get('status') != 'success':
        raise RuntimeError(f'CPU cache audit failed: {receipt}')
    if not receipt.get('receipts'):
        raise RuntimeError('CPU cache audit published no passing receipt')
    for row in receipt['receipts']:
        print('TRACK AUDIT PASSED - ' + row['colmap_fingerprint'])
    print('The cache is ready for the pinned A100 notebook.')
except BaseException as exc:
    failure = exc
    print(f'CPU audit ended with {type(exc).__name__}: {exc}')
    traceback.print_exception(type(exc), exc, exc.__traceback__)
finally:
    print('Flushing outstanding Google Drive writes...')
    try:
        drive.flush_and_unmount()
    except BaseException as flush_error:
        print(f'Drive flush/unmount failed: {flush_error}')
    print('Releasing the CPU Colab runtime now.')
    try:
        runtime.unassign()
    except BaseException as release_error:
        print(f'Runtime release request failed: {release_error}')
        if failure is None:
            failure = release_error
if failure is not None:
    raise failure
